<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6 — Quantitative Nanomechanics: Python Exercises

**SPM Syllabus 2026** · Interactive simulations for AFM contact mechanics and modulus extraction

These exercises accompany Chapter 6 of the textbook. Each exercise builds an interactive simulation that lets you explore how AFM force–indentation curves are converted into **quantitative material properties** using contact mechanics models (Hertz, Sneddon, JKR, DMT) and how the choice of model, tip geometry, and sample geometry affect the extracted Young's modulus.

**How to use this notebook:**
1. Run the first two cells (imports + widgets) before anything else.
2. Each exercise has a *theory cell* (markdown) followed by an *interactive code cell*.
3. Use the sliders to explore — the goal is to build physical intuition, not just run code.
4. At the end of each exercise, there is a short **Reflection** prompt — try to answer it before moving on.

**Chapter sections covered:**

| Exercise | Section | Topic |
|----------|---------|-------|
| 1 | 6.1 | Hertz model fitting — extract $E^*$ from $F(\delta)$ |
| 2 | 6.2, 6.4 | Tip geometry: spherical (Hertz) vs conical (Sneddon) |
| 3 | 6.3 | Adhesion: Hertz vs JKR vs DMT and the Tabor parameter |
| 4 | 6.5 | Substrate effects on thin samples (Bottom-Effect Correction) |
| 5 | 6.7 | Modulus mapping of a heterogeneous sample |
| 6 | 6.7 | Biological case: control vs stiffened cells (statistical comparison) |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy import stats

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
})

# Physical constants
NM = 1e-9   # nanometre  (m)
KPA = 1e3   # kilopascal (Pa)
NN = 1e-9   # nanonewton (N)


In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown, Checkbox
except ImportError:
    !pip install ipywidgets -q
    from ipywidgets import interact, FloatSlider, IntSlider, FloatLogSlider, Dropdown, Checkbox


---
## 1. Hertz Model Fitting — Extracting $E^*$ from $F(\delta)$

The **Hertz model** (Section 6.1) describes elastic contact between a spherical tip of radius $R$ and a flat sample:

$$F = \frac{4}{3}\, E^{*}\, \sqrt{R}\; \delta^{3/2}$$

where $E^{*}$ is the **effective (reduced) Young's modulus** that accounts for the elasticity of both tip and sample:

$$\frac{1}{E^{*}} = \frac{1 - \nu_{s}^{2}}{E_{s}} + \frac{1 - \nu_{t}^{2}}{E_{t}}$$

For a stiff tip on a soft sample ($E_t \gg E_s$), $E^{*} \approx E_{s}/(1-\nu_{s}^{2})$.

### Exercise

The function below generates **synthetic Hertz data** with known $E^*$, adds realistic experimental noise, and fits the model back to recover $E^*$. The third panel shows how the fitted modulus depends on the *fit-range limit* — a critical practical question: *over which indentation range should we fit?*

### Tasks
1. Set noise to 0 and confirm that the fit recovers $E^*$ exactly.
2. Increase noise to 20 % and observe how the standard error grows.
3. Vary the fit-range limit. At what range does the fitted $E^*$ become unstable? Why?

**Reflection:** *If a Hertz fit gives a modulus that depends strongly on the fit range, what does that tell you about the assumptions of the model?*


In [ ]:
def interactive_hertz_fit(E_true_kPa=20.0, R_nm=25.0, nu=0.5, noise_pct=5.0,
                          fit_max_nm=30.0, n_points=200):
    """Fit the Hertz model to noisy synthetic force-indentation data."""
    R       = R_nm * NM
    E_true  = E_true_kPa * KPA
    Es_true = E_true * (1 - nu**2)   # underlying sample Young's modulus

    # 1) Generate true curve
    delta = np.linspace(0.5, 50, n_points)          # nm
    delta_m = delta * NM
    F_true = (4/3) * E_true * np.sqrt(R) * delta_m**1.5 / NN   # nN

    # 2) Add noise
    rng = np.random.default_rng(7)
    noise_amp = noise_pct/100 * np.max(F_true)
    F_noisy = F_true + noise_amp * rng.normal(size=delta.size)

    # 3) Fit only over [0, fit_max_nm]
    def hertz(d_nm, E_Pa):
        return (4/3) * E_Pa * np.sqrt(R) * (d_nm*NM)**1.5 / NN

    mask = delta <= fit_max_nm
    try:
        popt, pcov = curve_fit(hertz, delta[mask], F_noisy[mask], p0=[E_true])
        E_fit, E_err = popt[0], np.sqrt(pcov[0, 0])
    except Exception:
        E_fit, E_err = np.nan, np.nan

    F_fit = hertz(delta, E_fit) if np.isfinite(E_fit) else np.zeros_like(delta)
    rmse  = np.sqrt(np.mean((F_noisy[mask] - F_fit[mask])**2))

    # 4) Stability sweep across fit ranges
    ranges = np.arange(5, 51, 2)
    E_sweep = []
    for r in ranges:
        m = delta <= r
        if m.sum() > 4:
            try:
                p, _ = curve_fit(hertz, delta[m], F_noisy[m], p0=[E_true])
                E_sweep.append(p[0]/KPA)
            except Exception:
                E_sweep.append(np.nan)
        else:
            E_sweep.append(np.nan)

    print("  Hertz Model Fitting")
    print("  " + "─"*44)
    print(f"  True modulus      E*  = {E_true_kPa:>7.2f} kPa")
    print(f"  Fitted modulus    E*  = {E_fit/KPA:>7.2f} ± {E_err/KPA:.2f} kPa")
    print(f"  Sample Young's modulus E_s = {Es_true/KPA:>7.2f} kPa  (ν = {nu})")
    print(f"  Relative error        = {(E_fit/E_true - 1)*100:+.1f} %")
    print(f"  Fit range             = 0 – {fit_max_nm:.0f} nm   ({mask.sum()} points)")
    print(f"  Residual RMSE         = {rmse:.3f} nN")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].scatter(delta, F_noisy, s=8, color='gray', alpha=0.5, label='Noisy data')
    axes[0].plot(delta, F_true, 'k-', lw=1, alpha=0.4, label=f'True (E*={E_true_kPa:.1f} kPa)')
    axes[0].plot(delta, F_fit, 'b-', lw=2.2, label=f'Fit (E*={E_fit/KPA:.1f} kPa)')
    axes[0].axvline(fit_max_nm, color='red', ls='--', lw=1, label='Fit range limit')
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Force–indentation fit')
    axes[0].legend(fontsize=8)

    axes[1].scatter(delta[mask], F_noisy[mask] - F_fit[mask], s=10, color='blue', alpha=0.5)
    axes[1].axhline(0, color='gray', ls=':', lw=1)
    axes[1].set_xlabel('Indentation δ (nm)')
    axes[1].set_ylabel('Residual (nN)')
    axes[1].set_title(f'Residuals (RMSE = {rmse:.3f} nN)')

    axes[2].plot(ranges, E_sweep, 'b-o', lw=2, ms=4)
    axes[2].axhline(E_true_kPa, color='green', ls='--', lw=1.5, label=f'True E* = {E_true_kPa:.1f} kPa')
    axes[2].axvline(fit_max_nm, color='red', ls='--', lw=1, label='current limit')
    axes[2].set_xlabel('Fit range limit (nm)')
    axes[2].set_ylabel('Fitted E* (kPa)')
    axes[2].set_title('Modulus stability vs fit range')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_hertz_fit,
    E_true_kPa=FloatSlider(value=20, min=1, max=200, step=1, description='E* true (kPa)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)'),
    nu=FloatSlider(value=0.5, min=0.1, max=0.49999, step=0.05, description='ν (Poisson)'),
    noise_pct=FloatSlider(value=5, min=0, max=30, step=1, description='Noise (%)'),
    fit_max_nm=FloatSlider(value=30, min=5, max=50, step=1, description='Fit range (nm)'),
    n_points=IntSlider(value=200, min=50, max=500, step=50, description='# points'),
);


---
## 2. Tip Geometry — Hertz vs Sneddon and the Cost of the Wrong Model

The **Sneddon model** (Section 6.2) describes a *conical* indenter with half-opening angle $\alpha$:

$$F = \frac{2}{\pi}\, E^{*}\, \tan(\alpha)\; \delta^{2}$$

The exponent in $F(\delta)$ is **set by the contact geometry**, not the material:

| Tip | Force law | Exponent | When it dominates |
|-----|-----------|----------|-------------------|
| Sphere (Hertz) | $F\propto\delta^{3/2}$ | 1.5 | $\delta \ll R$ (apex regime) |
| Cone (Sneddon) | $F\propto\delta^{2}$ | 2.0 | $\delta \gg R$ (pyramid body) |

If you fit Sneddon-regime data with the Hertz formula (or vice-versa), you get a **systematically wrong** modulus.

### Exercise

This simulation generates "true" data for a chosen tip geometry, then fits **both** Hertz and Sneddon models to it. The right panel shows the local exponent of $\log F$ vs $\log \delta$, which is the cleanest diagnostic of which model is appropriate.

### Tasks
1. Generate **spherical** data and fit with both models. Which exponent does the log–log slope reveal?
2. Switch to **conical** and look at the bias in the *Hertz-fitted* modulus.
3. Set $R = 5$ nm with $\delta_{\max} = 50$ nm — does either model fit cleanly across the whole curve?

**Reflection:** *Why is the log–log slope a more reliable indicator of tip geometry than the absolute modulus value?*


In [ ]:
def interactive_geometry(true_geom='Sphere', E_true_kPa=20.0, R_nm=25.0,
                         alpha_deg=18.0, delta_max_nm=50.0, noise_pct=3.0):
    """Compare Hertz vs Sneddon fits — including using the wrong model."""
    R = R_nm * NM
    E_true = E_true_kPa * KPA
    alpha = np.radians(alpha_deg)

    delta = np.linspace(0.5, delta_max_nm, 300)     # nm
    delta_m = delta * NM

    if true_geom == 'Sphere':
        F_true = (4/3) * E_true * np.sqrt(R) * delta_m**1.5 / NN
    else:
        F_true = (2/np.pi) * E_true * np.tan(alpha) * delta_m**2 / NN

    rng = np.random.default_rng(11)
    noise_amp = noise_pct/100 * np.max(F_true)
    F = F_true + noise_amp * rng.normal(size=delta.size)

    # Fit Hertz (spherical)
    def hertz(d_nm, E):
        return (4/3) * E * np.sqrt(R) * (d_nm*NM)**1.5 / NN
    def sneddon(d_nm, E):
        return (2/np.pi) * E * np.tan(alpha) * (d_nm*NM)**2 / NN

    try:
        E_h, = curve_fit(hertz, delta, F, p0=[E_true])[0]
    except Exception:
        E_h = np.nan
    try:
        E_s, = curve_fit(sneddon, delta, F, p0=[E_true])[0]
    except Exception:
        E_s = np.nan

    # Local log-log slope (smoothed)
    pos = (delta > 1) & (F > 0.05*np.max(F))
    log_d = np.log(delta[pos])
    log_F = np.log(F[pos])
    slope_global, intercept = np.polyfit(log_d, log_F, 1)

    # Sliding window slope
    win = 30
    centers, slopes = [], []
    for k in range(0, len(log_d)-win, 5):
        s, _ = np.polyfit(log_d[k:k+win], log_F[k:k+win], 1)
        slopes.append(s)
        centers.append(np.exp(np.mean(log_d[k:k+win])))

    print("  Hertz vs Sneddon — fitting the same curve with both models")
    print("  " + "─"*55)
    print(f"  True geometry     : {true_geom}   "
          f"({'sphere R=%.0f nm' % R_nm if true_geom=='Sphere' else 'cone α=%.0f°' % alpha_deg})")
    print(f"  True modulus  E*  = {E_true_kPa:.1f} kPa")
    print()
    print(f"  {'Fit model':<18s}  {'E* (kPa)':>10s}  {'Bias':>8s}")
    print(f"  {'─'*18}  {'─'*10}  {'─'*8}")
    print(f"  {'Hertz (sphere)':<18s}  {E_h/KPA:>10.1f}  {(E_h/E_true-1)*100:>+7.1f}%")
    print(f"  {'Sneddon (cone)':<18s}  {E_s/KPA:>10.1f}  {(E_s/E_true-1)*100:>+7.1f}%")
    print()
    print(f"  Global log-log slope ≈ {slope_global:.2f}   "
          f"(expected: 1.5 sphere, 2.0 cone)")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].scatter(delta, F, s=8, color='gray', alpha=0.5, label='Noisy data')
    axes[0].plot(delta, hertz(delta, E_h), 'b-', lw=2,
                 label=f'Hertz fit ({E_h/KPA:.1f} kPa)')
    axes[0].plot(delta, sneddon(delta, E_s), 'r-', lw=2,
                 label=f'Sneddon fit ({E_s/KPA:.1f} kPa)')
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title(f'True geometry: {true_geom}')
    axes[0].legend(fontsize=8)

    axes[1].loglog(delta[pos], F[pos], 'o', ms=3, color='gray', alpha=0.5, label='Data')
    axes[1].loglog(delta[pos], np.exp(intercept) * delta[pos]**slope_global,
                   'k--', lw=1.5, label=f'slope = {slope_global:.2f}')
    axes[1].loglog(delta[pos], delta[pos]**1.5 * F[pos][0]/(delta[pos][0]**1.5),
                   'b:', lw=1.2, label='slope 1.5 (Hertz)')
    axes[1].loglog(delta[pos], delta[pos]**2.0 * F[pos][0]/(delta[pos][0]**2.0),
                   'r:', lw=1.2, label='slope 2.0 (Sneddon)')
    axes[1].set_xlabel('δ (nm)')
    axes[1].set_ylabel('F (nN)')
    axes[1].set_title('log–log slope diagnostic')
    axes[1].legend(fontsize=8)

    axes[2].plot(centers, slopes, 'k-o', ms=3)
    axes[2].axhline(1.5, color='blue', ls='--', lw=1, label='Hertz')
    axes[2].axhline(2.0, color='red', ls='--', lw=1, label='Sneddon')
    axes[2].set_xscale('log')
    axes[2].set_xlabel('δ (nm)')
    axes[2].set_ylabel('Local exponent')
    axes[2].set_title('Local exponent of F(δ)')
    axes[2].legend(fontsize=8)
    axes[2].set_ylim(1.0, 2.5)

    plt.tight_layout()
    plt.show()

interact(
    interactive_geometry,
    true_geom=Dropdown(options=['Sphere', 'Cone'], value='Sphere', description='True geom.'),
    E_true_kPa=FloatSlider(value=20, min=1, max=200, step=1, description='E* true (kPa)'),
    R_nm=FloatSlider(value=25, min=2, max=100, step=1, description='R_tip (nm)'),
    alpha_deg=FloatSlider(value=18, min=5, max=45, step=1, description='Cone α (°)'),
    delta_max_nm=FloatSlider(value=50, min=10, max=200, step=5, description='δ_max (nm)'),
    noise_pct=FloatSlider(value=3, min=0, max=20, step=1, description='Noise (%)'),
);


---
## 3. Adhesion — Hertz vs JKR vs DMT and the Tabor Parameter

When adhesion is significant, the **Hertz model is no longer adequate** (Section 6.3). Two limiting adhesive models exist:

| Model | Assumption | Pull-off force | Best for |
|-------|------------|----------------|----------|
| **JKR** | Adhesion *inside* contact area | $F_{\mathrm{pull}} = -\tfrac{3}{2}\pi R W$ | Soft, strongly adhesive (cells, polymers) |
| **DMT** | Adhesion *outside* contact area | $F_{\mathrm{pull}} = -2\pi R W$ | Stiff, weakly adhesive (ceramics, dry contacts) |

The **Tabor parameter** decides which regime applies:

$$\mu = \left(\frac{R\,W^{2}}{E^{*2}\,z_{0}^{3}}\right)^{1/3}$$

with $z_0 \approx 0.3{-}0.5$ nm. For $\mu \gg 1$ → JKR; $\mu \ll 1$ → DMT; $\mu \sim 1$ → transition.

### Exercise

This simulation generates the *true* approach curve as a weighted blend of JKR-like and DMT-like behaviour (the weight depends on $\mu$), then fits Hertz, JKR-like, and DMT models. The third panel shows the Tabor map with your current operating point marked.

### Tasks
1. For a soft cell ($E^* = 5$ kPa, $W = 50$ mJ/m²), where does the Tabor parameter sit? Which model should you use?
2. Compare the **Hertz-fitted** modulus to the true value when adhesion is strong. Is Hertz an over- or under-estimator?
3. Vary $z_0$ between 0.1 and 1 nm. How sensitive is the model selection to this parameter?

**Reflection:** *Cell biology AFM almost always lives in the JKR regime. Why does this make Hertz fits inherently biased for cell mechanics?*


In [ ]:
def interactive_contact_models(E_kPa=20.0, gamma_mJ_m2=30.0, R_nm=25.0, z0_nm=0.3,
                               noise_pct=2.0):
    """Compare Hertz, JKR-like and DMT fits, and locate the Tabor regime."""
    R       = R_nm * NM
    E_star  = E_kPa * KPA
    gamma   = gamma_mJ_m2 * 1e-3        # J/m^2  (= N/m)
    z0      = z0_nm * NM

    # Tabor parameter μ_T
    mu_T = (R * gamma**2 / (E_star**2 * z0**3))**(1/3)

    # Pull-off forces (analytical limits)
    F_pull_JKR = 1.5 * np.pi * R * gamma / NN     # nN  (3/2 π R W)
    F_pull_DMT = 2.0 * np.pi * R * gamma / NN     # nN  (2   π R W)

    delta = np.linspace(0, 40, 400)               # nm
    delta_m = delta * NM

    # Hertz reference (no adhesion)
    F_hertz = (4/3) * E_star * np.sqrt(R) * delta_m**1.5 / NN

    # JKR-like phenomenological:  Hertz(0.85 E*) − exponentially decaying adhesion well
    F_jkr   = (4/3) * E_star * 0.85 * np.sqrt(R) * delta_m**1.5 / NN               - F_pull_JKR * np.exp(-delta/3.0)

    # DMT: Hertz with a constant adhesion offset
    F_dmt   = F_hertz - F_pull_DMT

    # Observed data is a μ-weighted blend
    w_jkr = np.clip(mu_T / 5.0, 0, 1)
    F_data = w_jkr * F_jkr + (1 - w_jkr) * F_dmt
    rng    = np.random.default_rng(42)
    F_data += noise_pct/100 * np.max(np.abs(F_data)) * rng.normal(size=delta.size)

    # Fits restricted to repulsive part (avoid pull-off well)
    fit_m = delta > 5

    def hertz(d, E):
        return (4/3) * E * np.sqrt(R) * (d*NM)**1.5 / NN
    def dmt(d, E):
        return (4/3) * E * np.sqrt(R) * (d*NM)**1.5 / NN - F_pull_DMT

    try:
        E_h, = curve_fit(hertz, delta[fit_m], F_data[fit_m], p0=[E_star])[0]
    except Exception:
        E_h = np.nan
    try:
        E_d, = curve_fit(dmt, delta[fit_m], F_data[fit_m], p0=[E_star])[0]
    except Exception:
        E_d = np.nan

    if mu_T > 3:
        regime = "JKR (soft, strongly adhesive)"
    elif mu_T < 0.1:
        regime = "DMT (stiff, weakly adhesive)"
    else:
        regime = "Transition (Maugis–Dugdale)"

    print("  Hertz vs JKR vs DMT — same data, three models")
    print("  " + "─"*55)
    print(f"  E* = {E_kPa:.1f} kPa,  W = γ = {gamma_mJ_m2:.0f} mJ/m²,  R = {R_nm:.0f} nm,  z₀ = {z0_nm:.2f} nm")
    print(f"  Tabor parameter   μ = {mu_T:.3f}   →  {regime}")
    print()
    print(f"  {'Model':<14s}  {'E* (kPa)':>10s}  {'F_pull (nN)':>14s}")
    print(f"  {'─'*14}  {'─'*10}  {'─'*14}")
    print(f"  {'Hertz':<14s}  {E_h/KPA:>10.1f}  {'—':>14s}")
    print(f"  {'DMT':<14s}  {E_d/KPA:>10.1f}  {-F_pull_DMT:>14.2f}")
    print(f"  {'JKR':<14s}  {'—':>10s}  {-F_pull_JKR:>14.2f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].scatter(delta, F_data, s=6, color='gray', alpha=0.4, label='Data')
    axes[0].plot(delta, hertz(delta, E_h), 'b-', lw=2,
                 label=f'Hertz ({E_h/KPA:.1f} kPa)')
    axes[0].plot(delta, F_jkr, 'g-', lw=2, label='JKR-like (true)')
    axes[0].plot(delta, dmt(delta, E_d), 'r-', lw=2,
                 label=f'DMT ({E_d/KPA:.1f} kPa)')
    axes[0].axhline(0, color='gray', ls=':', lw=0.8)
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Three models, one dataset')
    axes[0].legend(fontsize=8)

    models = ['Hertz fit', 'DMT fit', 'True E*']
    vals   = [E_h/KPA, E_d/KPA, E_kPa]
    cols   = ['#2F6DB3', '#C84A4A', '#333333']
    axes[1].bar(models, vals, color=cols, edgecolor='white', width=0.55)
    for i, v in enumerate(vals):
        axes[1].text(i, v + 0.03*max(vals), f'{v:.1f}', ha='center', fontsize=9)
    axes[1].set_ylabel("E* (kPa)")
    axes[1].set_title('Model choice → different E*')

    # Tabor parameter map
    E_grid_axis = np.logspace(2, 6, 120)
    G_grid_axis = np.logspace(-3, 0, 120)
    EG, GG = np.meshgrid(E_grid_axis, G_grid_axis)
    mu_grid = (R * GG**2 / (EG**2 * z0**3))**(1/3)
    pcm = axes[2].pcolormesh(EG/KPA, GG*1e3, np.log10(mu_grid),
                             cmap='RdYlBu_r', shading='auto', vmin=-2, vmax=2)
    axes[2].plot(E_kPa, gamma_mJ_m2, 'k*', ms=18, zorder=5,
                 label=f'μ = {mu_T:.2f}')
    axes[2].contour(EG/KPA, GG*1e3, np.log10(mu_grid),
                    levels=[-1, 0, 1], colors='k', linewidths=0.7)
    axes[2].set_xscale('log'); axes[2].set_yscale('log')
    axes[2].set_xlabel('E* (kPa)'); axes[2].set_ylabel('γ (mJ/m²)')
    axes[2].set_title('Tabor regime map (DMT ← μ → JKR)')
    fig.colorbar(pcm, ax=axes[2], label='log₁₀(μ)')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(
    interactive_contact_models,
    E_kPa=FloatLogSlider(value=20, base=10, min=0, max=4, step=0.1, description='E* (kPa)'),
    gamma_mJ_m2=FloatSlider(value=30, min=1, max=200, step=5, description='γ (mJ/m²)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)'),
    z0_nm=FloatSlider(value=0.3, min=0.1, max=1.0, step=0.05, description='z₀ (nm)'),
    noise_pct=FloatSlider(value=2, min=0, max=15, step=1, description='Noise (%)'),
);


---
## 4. Substrate Effects on Thin Samples — The 10 % Rule

Real biological samples (cells, biofilms, hydrogels) have **finite thickness** $h$. As the tip indents, the deformation field eventually feels the rigid substrate underneath, and the measurement becomes a **composite** of the sample and the substrate (Section 6.5).

A simple **Bottom-Effect Correction** (Dimitriadis-style) for a thin layer of thickness $h$ on a rigid substrate, indented by a sphere of radius $R$, gives an enhanced apparent stiffness:

$$F_{\mathrm{thin}}(\delta) \approx F_{\mathrm{Hertz}}(\delta)\, \Big[\,1 + 1.133\,\chi + 1.283\,\chi^{2} + 0.769\,\chi^{3} + 0.0975\,\chi^{4}\,\Big]$$

with $\chi = \sqrt{R\,\delta}\,/\,h$. The bracket → 1 when $h \gg \sqrt{R\delta}$ (semi-infinite limit).

The familiar rule of thumb is the **10 % rule**: keep $\delta / h < 0.1$.

### Exercise

This simulation generates a thin-sample force curve, then fits **plain Hertz** to it. The right panel sweeps the fit-range limit and shows when the Hertz fit starts overestimating the true modulus.

### Tasks
1. Set $h = 5$ µm (cells), $E^* = 5$ kPa, $R = 25$ nm. How shallow must $\delta$ be to recover $E^*$ within 10 %?
2. Reduce $h$ to 500 nm (a thin biofilm). Is the 10 % rule still safe?
3. Increase $R$ to 1 µm (colloidal probe). Does the 10 % rule still apply? Why does $\chi$ depend on $\sqrt{R}$?

**Reflection:** *Why is "use a colloidal probe to reduce stress" not a free lunch when the sample is thin?*


In [ ]:
def interactive_substrate(E_true_kPa=10.0, R_nm=25.0, h_um=5.0,
                          delta_max_nm=200.0, noise_pct=2.0, fit_max_nm=80.0):
    """Substrate effect on a thin elastic sample (Dimitriadis correction)."""
    R       = R_nm * NM
    h       = h_um * 1e-6
    E_true  = E_true_kPa * KPA

    delta   = np.linspace(0.5, delta_max_nm, 400)   # nm
    delta_m = delta * NM
    chi     = np.sqrt(R * delta_m) / h              # dimensionless

    # Hertz semi-infinite reference
    F_inf   = (4/3) * E_true * np.sqrt(R) * delta_m**1.5 / NN

    # Bottom-effect correction (Dimitriadis 2002, bonded case)
    bec = 1 + 1.133*chi + 1.283*chi**2 + 0.769*chi**3 + 0.0975*chi**4
    F_thin = F_inf * bec

    rng = np.random.default_rng(3)
    F_obs = F_thin + noise_pct/100 * np.max(F_thin) * rng.normal(size=delta.size)

    # Fit plain Hertz to first portion
    def hertz(d, E):
        return (4/3) * E * np.sqrt(R) * (d*NM)**1.5 / NN

    mask = delta <= fit_max_nm
    try:
        E_apparent, = curve_fit(hertz, delta[mask], F_obs[mask], p0=[E_true])[0]
    except Exception:
        E_apparent = np.nan
    err_pct = (E_apparent/E_true - 1) * 100

    # Sweep fit range to plot apparent E* vs δ_max
    ranges = np.arange(10, int(delta_max_nm)+1, 5)
    E_app_sweep = []
    for r in ranges:
        m = delta <= r
        if m.sum() > 4:
            try:
                p, _ = curve_fit(hertz, delta[m], F_obs[m], p0=[E_true])
                E_app_sweep.append(p[0]/KPA)
            except Exception:
                E_app_sweep.append(np.nan)
        else:
            E_app_sweep.append(np.nan)
    E_app_sweep = np.array(E_app_sweep)

    # Find δ where apparent E* exceeds true by 10 %
    over10 = ranges[E_app_sweep > 1.10*E_true_kPa]
    crit_delta = over10[0] if over10.size else None

    print("  Substrate effect on a thin elastic layer")
    print("  " + "─"*48)
    print(f"  Sample thickness   h = {h_um:.2f} µm")
    print(f"  True modulus      E* = {E_true_kPa:.1f} kPa")
    print(f"  Tip radius         R = {R_nm:.0f} nm")
    print(f"  Fit range            = 0 – {fit_max_nm:.0f} nm")
    print(f"  Apparent (Hertz)  E* = {E_apparent/KPA:.1f} kPa  ({err_pct:+.1f} %)")
    if crit_delta is not None:
        print(f"  → Hertz fit exceeds 10% bias at δ_max ≈ {crit_delta} nm  "
              f"(δ/h = {crit_delta*NM/h*100:.1f}%)")
    print(f"  10% rule limit:  δ ≤ 0.1 h = {0.1*h_um*1000:.0f} nm")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    axes[0].plot(delta, F_inf, 'k--', lw=1.4, alpha=0.6, label='Semi-infinite Hertz')
    axes[0].plot(delta, F_thin, 'r-', lw=2, label=f'Thin layer (h={h_um:.1f} µm)')
    axes[0].scatter(delta, F_obs, s=4, color='gray', alpha=0.4, label='Noisy data')
    axes[0].axvline(fit_max_nm, color='blue', ls=':', lw=1.2, label='fit range')
    axes[0].axvline(0.1*h_um*1000, color='green', ls=':', lw=1.2, label='10% rule')
    axes[0].set_xlabel('Indentation δ (nm)')
    axes[0].set_ylabel('Force (nN)')
    axes[0].set_title('Thin-sample stiffening')
    axes[0].legend(fontsize=8)

    axes[1].plot(delta, bec, 'r-', lw=2)
    axes[1].axhline(1, color='gray', ls=':')
    axes[1].axvline(0.1*h_um*1000, color='green', ls=':', lw=1.2, label='10% rule')
    axes[1].set_xlabel('δ (nm)')
    axes[1].set_ylabel('Stiffening factor BEC(δ)')
    axes[1].set_title(f'BEC factor  (χ = √(Rδ)/h)')
    axes[1].legend(fontsize=8)

    axes[2].plot(ranges, E_app_sweep, 'b-o', ms=3)
    axes[2].axhline(E_true_kPa, color='green', ls='--', lw=1.5,
                    label=f'True E* = {E_true_kPa:.1f} kPa')
    axes[2].axhline(1.1*E_true_kPa, color='orange', ls=':', lw=1, label='+10 %')
    axes[2].axvline(0.1*h_um*1000, color='green', ls=':', lw=1.2, label='10% rule')
    axes[2].set_xlabel('Fit range δ_max (nm)')
    axes[2].set_ylabel('Apparent E* (kPa)')
    axes[2].set_title('Hertz overestimation vs δ_max')
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

interact(
    interactive_substrate,
    E_true_kPa=FloatSlider(value=10, min=1, max=100, step=1, description='E* (kPa)'),
    R_nm=FloatSlider(value=25, min=5, max=1000, step=25, description='R_tip (nm)'),
    h_um=FloatSlider(value=5.0, min=0.2, max=20.0, step=0.2, description='h (µm)'),
    delta_max_nm=FloatSlider(value=200, min=50, max=500, step=25, description='δ_max (nm)'),
    fit_max_nm=FloatSlider(value=80, min=10, max=400, step=5, description='Fit range (nm)'),
    noise_pct=FloatSlider(value=2, min=0, max=15, step=1, description='Noise (%)'),
);


---
## 5. Modulus Mapping — From Curves to Mechanical Images

Force-mapping mode (Section 6.7) records a force curve at every pixel of a grid. Hertz fitting at each pixel produces a 2-D **modulus map** that resolves spatial heterogeneity of the sample.

### Exercise

Below, a synthetic sample contains a soft circular region (e.g. cytoplasm) on a stiffer background (e.g. cytoskeleton-rich periphery). Each pixel gets its own noisy force curve, the Hertz fit is performed, and the resulting modulus map is reconstructed.

### Tasks
1. Vary the contrast $E_{\text{hard}}/E_{\text{soft}}$. Below which contrast does the soft region get lost in noise?
2. Increase the noise. The histogram of fitted moduli broadens — when does the bimodal nature of the sample disappear?
3. Reduce the grid size. How few pixels can you afford while still resolving the soft region?

**Reflection:** *Why is the modulus histogram (right-most panel) often as informative as the spatial map itself?*


In [ ]:
def interactive_modulus_map(N=24, E_soft_kPa=5.0, E_hard_kPa=50.0, R_nm=25.0,
                            noise_pct=8.0, n_points_per_curve=80, delta_max_nm=30.0):
    """Generate a force-volume map and reconstruct the modulus image."""
    R = R_nm * NM
    rng = np.random.default_rng(42)

    x = np.arange(N); y = np.arange(N)
    X, Y = np.meshgrid(x, y)
    cx, cy = N/2, N/2
    is_soft = np.sqrt((X-cx)**2 + (Y-cy)**2) < N/3

    # Local true modulus at each pixel (with biological variability)
    E_true_map = np.where(is_soft,
                          E_soft_kPa * (1 + 0.10*rng.normal(size=(N, N))),
                          E_hard_kPa * (1 + 0.05*rng.normal(size=(N, N))))
    E_true_map = np.clip(E_true_map, 0.5, None)

    delta = np.linspace(0.5, delta_max_nm, n_points_per_curve)
    delta_m = delta * NM

    def hertz(d, E):
        return (4/3) * E * np.sqrt(R) * (d*NM)**1.5 / NN

    E_fit_map = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            E_loc = E_true_map[i, j] * KPA
            F_true = (4/3) * E_loc * np.sqrt(R) * delta_m**1.5 / NN
            F_obs = F_true + noise_pct/100 * np.max(F_true) * rng.normal(size=delta.size)
            try:
                E_hat, = curve_fit(hertz, delta, F_obs, p0=[E_loc])[0]
                E_fit_map[i, j] = E_hat / KPA
            except Exception:
                E_fit_map[i, j] = np.nan

    # Statistics
    E_fit_soft = E_fit_map[is_soft]
    E_fit_hard = E_fit_map[~is_soft]

    print("  Force-volume modulus mapping")
    print("  " + "─"*44)
    print(f"  Grid              = {N} × {N}  ({N*N} curves)")
    print(f"  Tip radius        = {R_nm:.0f} nm")
    print(f"  Noise per curve   = {noise_pct:.0f} %")
    print()
    print(f"  {'Region':<10s}  {'True E* (kPa)':>15s}  {'Fitted E* (kPa)':>20s}  {'σ':>8s}")
    print(f"  {'─'*10}  {'─'*15}  {'─'*20}  {'─'*8}")
    print(f"  {'Soft':<10s}  {E_soft_kPa:>15.1f}  "
          f"{np.nanmean(E_fit_soft):>15.1f}        {np.nanstd(E_fit_soft):>5.1f}")
    print(f"  {'Hard':<10s}  {E_hard_kPa:>15.1f}  "
          f"{np.nanmean(E_fit_hard):>15.1f}        {np.nanstd(E_fit_hard):>5.1f}")
    print(f"  Apparent contrast = {np.nanmean(E_fit_hard)/np.nanmean(E_fit_soft):.2f}× "
          f"(true {E_hard_kPa/E_soft_kPa:.2f}×)")

    fig, axes = plt.subplots(1, 4, figsize=(17, 4))

    im0 = axes[0].imshow(is_soft.astype(int), cmap='coolwarm_r', origin='lower',
                          vmin=0, vmax=1)
    axes[0].set_title('Sample structure'); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

    im1 = axes[1].imshow(E_true_map, cmap='viridis', origin='lower')
    axes[1].set_title('True E* (kPa)')
    fig.colorbar(im1, ax=axes[1], shrink=0.8)

    im2 = axes[2].imshow(E_fit_map, cmap='viridis', origin='lower',
                          vmin=np.nanmin(E_true_map), vmax=np.nanmax(E_true_map))
    axes[2].set_title('Fitted E* map')
    fig.colorbar(im2, ax=axes[2], shrink=0.8)

    bins = np.linspace(np.nanmin(E_fit_map), np.nanmax(E_fit_map), 30)
    axes[3].hist(E_fit_soft, bins=bins, alpha=0.6, color='steelblue', label='Soft')
    axes[3].hist(E_fit_hard, bins=bins, alpha=0.6, color='firebrick', label='Hard')
    axes[3].axvline(E_soft_kPa, color='steelblue', ls='--')
    axes[3].axvline(E_hard_kPa, color='firebrick',  ls='--')
    axes[3].set_xlabel('Fitted E* (kPa)'); axes[3].set_ylabel('# pixels')
    axes[3].set_title('Modulus distribution')
    axes[3].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(
    interactive_modulus_map,
    N=IntSlider(value=24, min=8, max=40, step=4, description='Grid N'),
    E_soft_kPa=FloatSlider(value=5, min=1, max=50, step=1, description='E* soft (kPa)'),
    E_hard_kPa=FloatSlider(value=50, min=5, max=500, step=5, description='E* hard (kPa)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)'),
    noise_pct=FloatSlider(value=8, min=0, max=40, step=1, description='Noise (%)'),
    n_points_per_curve=IntSlider(value=80, min=20, max=200, step=20, description='# pts/curve'),
    delta_max_nm=FloatSlider(value=30, min=10, max=100, step=5, description='δ_max (nm)'),
);


---
## 6. Biological Case Study — Control vs Stiffened Cells

Section 6.7 walks through the **mechanobiology of an osteoblast under shear stress**: cells respond to mechanical cues by reorganising their cytoskeleton, which AFM detects as a **right-shift in the modulus distribution**.

### Exercise

Below, two synthetic populations of cells are simulated:

* **Control** — cytoskeleton at baseline state, modulus log-normally distributed around $E_{\text{ctrl}}$.
* **Stiffened** — after shear stress, modulus shifts upward by a factor and the distribution broadens.

The simulation runs many "force-curve experiments" per cell, fits each one, and then performs a **statistical comparison** (Welch's $t$-test on log-transformed moduli + Mann–Whitney $U$ test, Cohen's $d$, log fold-change).

### Tasks
1. Bring the two populations close together. At what fold-change does $p > 0.05$? How much does $n_{\text{cells}}$ help?
2. Increase the within-cell variability. Notice how individual fits get noisier but the population means stay reliable.
3. Toggle the *log-normal* checkbox off → linear normal distribution. How does this change the inferred fold-change?

**Reflection:** *Cell-mechanics modulus distributions are typically log-normal. What does this imply for choosing between a $t$-test and a Mann–Whitney test? Why are box plots usually drawn on a log scale?*


In [ ]:
def interactive_cell_population(E_ctrl_kPa=2.0, fold_change=1.6, sigma_log=0.35,
                                n_cells=40, n_curves_per_cell=8,
                                noise_pct=8.0, log_normal=True, R_nm=25.0):
    """Simulate two cell populations and perform a statistical comparison."""
    R = R_nm * NM
    rng = np.random.default_rng(7)

    def sample_cell_modulus(E_mean, n):
        if log_normal:
            mu = np.log(E_mean) - 0.5*sigma_log**2
            return rng.lognormal(mean=mu, sigma=sigma_log, size=n)
        else:
            return np.maximum(E_mean + sigma_log*E_mean*rng.normal(size=n), 0.1)

    E_ctrl_cells   = sample_cell_modulus(E_ctrl_kPa,                n_cells)
    E_stiff_cells  = sample_cell_modulus(E_ctrl_kPa*fold_change,    n_cells)

    delta = np.linspace(0.5, 30, 60)
    delta_m = delta * NM

    def hertz(d, E):
        return (4/3) * E * np.sqrt(R) * (d*NM)**1.5 / NN

    def fit_population(E_cells):
        E_fit = []
        for E_true in E_cells:
            E_loc = E_true * KPA
            for _ in range(n_curves_per_cell):
                F_true = (4/3) * E_loc * np.sqrt(R) * delta_m**1.5 / NN
                F_obs  = F_true + noise_pct/100 * np.max(F_true) * rng.normal(size=delta.size)
                try:
                    E_hat, = curve_fit(hertz, delta, F_obs, p0=[E_loc])[0]
                    E_fit.append(E_hat / KPA)
                except Exception:
                    pass
        return np.array(E_fit)

    E_ctrl_fits  = fit_population(E_ctrl_cells)
    E_stiff_fits = fit_population(E_stiff_cells)

    # Statistics — log-transformed t-test + Mann-Whitney
    log_c = np.log(E_ctrl_fits[E_ctrl_fits > 0])
    log_s = np.log(E_stiff_fits[E_stiff_fits > 0])
    t_stat, p_t = stats.ttest_ind(log_c, log_s, equal_var=False)
    u_stat, p_u = stats.mannwhitneyu(E_ctrl_fits, E_stiff_fits, alternative='two-sided')

    # Cohen's d on log scale
    d_pool = np.sqrt((log_c.var(ddof=1) + log_s.var(ddof=1)) / 2)
    d_cohen = (log_s.mean() - log_c.mean()) / d_pool

    fold_obs = np.exp(log_s.mean() - log_c.mean())

    print("  Statistical comparison: control vs stiffened cells")
    print("  " + "─"*54)
    print(f"  n cells per group   = {n_cells}    n curves/cell = {n_curves_per_cell}")
    print(f"  Distribution        = {'log-normal' if log_normal else 'normal'}  (σ_log = {sigma_log:.2f})")
    print(f"  Imposed fold change = {fold_change:.2f}×")
    print(f"  Observed fold change= {fold_obs:.2f}×")
    print()
    print(f"  Median E* control   = {np.median(E_ctrl_fits):>6.2f} kPa "
          f"[{np.percentile(E_ctrl_fits,25):.2f}, {np.percentile(E_ctrl_fits,75):.2f}]")
    print(f"  Median E* stiffened = {np.median(E_stiff_fits):>6.2f} kPa "
          f"[{np.percentile(E_stiff_fits,25):.2f}, {np.percentile(E_stiff_fits,75):.2f}]")
    print()
    print(f"  Welch t-test (log)  : t = {t_stat:>+6.2f},  p = {p_t:.2e}")
    print(f"  Mann-Whitney U      : U = {u_stat:>6.0f},  p = {p_u:.2e}")
    print(f"  Cohen's d (log)     : {d_cohen:+.2f}  "
          f"({'small' if abs(d_cohen)<0.5 else 'medium' if abs(d_cohen)<0.8 else 'large'})")
    sig = 'YES ✓' if p_t < 0.05 else 'no'
    print(f"  Significant at α=5% : {sig}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    bins = np.logspace(np.log10(max(0.1, np.min([E_ctrl_fits.min(), E_stiff_fits.min()]))),
                       np.log10(np.max([E_ctrl_fits.max(), E_stiff_fits.max()])), 40)
    axes[0].hist(E_ctrl_fits,  bins=bins, alpha=0.55, color='steelblue', label='Control')
    axes[0].hist(E_stiff_fits, bins=bins, alpha=0.55, color='firebrick',  label='Stiffened')
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Fitted E* (kPa)')
    axes[0].set_ylabel('# curves')
    axes[0].set_title('Modulus distribution (log-x)')
    axes[0].legend(fontsize=9)

    box = axes[1].boxplot([E_ctrl_fits, E_stiff_fits],
                          labels=['Control', 'Stiffened'],
                          patch_artist=True, widths=0.5)
    for patch, c in zip(box['boxes'], ['steelblue', 'firebrick']):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    axes[1].set_yscale('log')
    axes[1].set_ylabel('E* (kPa, log)')
    axes[1].set_title(f'Box plot   p = {p_t:.1e}')

    axes[2].hist(np.log10(E_ctrl_fits),  bins=30, alpha=0.55, color='steelblue', label='Control')
    axes[2].hist(np.log10(E_stiff_fits), bins=30, alpha=0.55, color='firebrick',  label='Stiffened')
    axes[2].axvline(np.log10(E_ctrl_kPa), color='steelblue', ls='--')
    axes[2].axvline(np.log10(E_ctrl_kPa*fold_change), color='firebrick', ls='--')
    axes[2].set_xlabel('log₁₀ E* (kPa)')
    axes[2].set_ylabel('# curves')
    axes[2].set_title(f"Cohen's d = {d_cohen:+.2f}")
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

interact(
    interactive_cell_population,
    E_ctrl_kPa=FloatSlider(value=2.0, min=0.5, max=20, step=0.5, description='E_ctrl (kPa)'),
    fold_change=FloatSlider(value=1.6, min=1.0, max=4.0, step=0.05, description='Fold change'),
    sigma_log=FloatSlider(value=0.35, min=0.05, max=1.0, step=0.05, description='σ_log'),
    n_cells=IntSlider(value=40, min=5, max=100, step=5, description='# cells'),
    n_curves_per_cell=IntSlider(value=8, min=1, max=30, step=1, description='# curves/cell'),
    noise_pct=FloatSlider(value=8, min=0, max=30, step=1, description='Noise (%)'),
    R_nm=FloatSlider(value=25, min=5, max=100, step=5, description='R_tip (nm)'),
    log_normal=Checkbox(value=True, description='log-normal pop'),
);


---

## Summary

| Exercise | Section | Key concept | Take-home |
|---|---|---|---|
| 1 | 6.1 | Hertz fit | $E^*$ depends on noise, fit range, and Poisson assumption |
| 2 | 6.2, 6.4 | Hertz vs Sneddon | log–log slope reveals tip geometry; wrong geometry → biased $E^*$ |
| 3 | 6.3 | JKR / DMT / Tabor | Adhesion couples to deformation; Hertz biases the modulus in adhesive cell systems |
| 4 | 6.5 | Substrate effect | Thin samples appear stiffer; the 10 % rule keeps Hertz unbiased to ~10 % |
| 5 | 6.7 | Modulus mapping | Spatial heterogeneity is recovered, but noise sets the resolvable contrast |
| 6 | 6.7 | Cell populations | Mechanobiological shifts are detected via log-scale statistics |

**Key chapter take-aways:**
- AFM nanomechanics is a **model-based measurement**: the same $F(\delta)$ yields different $E^*$ values under different model assumptions.
- Geometry (sphere vs cone), adhesion (Hertz vs JKR vs DMT), and finite thickness (BEC) each introduce systematic biases.
- Quantitative reliability requires **explicit reporting** of: tip type and radius, indentation range, model used, fit quality, and substrate.
- For cells, statistical power lives in the **distribution shape** as much as the mean.

---

*End of Chapter 6 exercises. Chapter 7 will extend force spectroscopy to single molecules and unfolding/rupture events.*
